In [16]:
def zero_trust_authorize(request, resource_policy):
    reasons = []

    allowed_users = resource_policy.get(
        request["resource"],
        set(),
    )

    if request["user"] not in allowed_users:
        reasons.append(
            "user not authorized for this resource"
        )

    if not request["mfa_passed"]:
        reasons.append("MFA not completed")

    if not request["device_posture"]["antivirus_enabled"]:
        reasons.append("antivirus disabled")

    if not request["device_posture"]["os_patched"]:
        reasons.append("OS not fully patched")

    return {
        "granted": len(reasons) == 0,
        "reasons": reasons,
    }


policy = {
    "finance_db": {
        "csmith",
        "afinance",
    }
}

healthy_request = {
    "user": "csmith",
    "mfa_passed": True,
    "device_posture": {
        "antivirus_enabled": True,
        "os_patched": True,
    },
    "resource": "finance_db",
}

unhealthy_request = {
    "user": "csmith",
    "mfa_passed": True,
    "device_posture": {
        "antivirus_enabled": False,
        "os_patched": True,
    },
    "resource": "finance_db",
}

unauthorized_request = {
    "user": "attacker99",
    "mfa_passed": True,
    "device_posture": {
        "antivirus_enabled": True,
        "os_patched": True,
    },
    "resource": "finance_db",
}

print("Healthy Request")
print(zero_trust_authorize(
    healthy_request,
    policy,
))

print()

print("Unhealthy Device")
print(zero_trust_authorize(
    unhealthy_request,
    policy,
))

print()

print("Unauthorized User")
print(zero_trust_authorize(
    unauthorized_request,
    policy,
))

Healthy Request
{'granted': True, 'reasons': []}

Unhealthy Device
{'granted': False, 'reasons': ['antivirus disabled']}

Unauthorized User
{'granted': False, 'reasons': ['user not authorized for this resource']}
